# CIFAR-10 Image Classification using Convolutional Neural Networks (CNN)

## Objective
The objective of this notebook is to build, train, and evaluate a **Convolutional Neural Network (CNN)** for classifying images from the CIFAR-10 dataset.

The notebook covers:
- Loading the CIFAR-10 dataset
- Preprocessing image data
- Building a CNN architecture
- Compiling and training the model
- Evaluating model performance
- Making predictions on unseen images

# Importing Required Libraries

This section imports all the necessary libraries used throughout the notebook.

The imported modules provide functionality for:

- Numerical computations using NumPy
- Building deep learning models using TensorFlow/Keras
- Loading the CIFAR-10 dataset
- Creating CNN layers
- Performing data augmentation (optional)
- Visualizing results using Matplotlib and Seaborn

In [3]:
import numpy as np
from tensorflow import keras
from keras import Input
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    Activation,
    Flatten,
    RandomFlip,
    RandomZoom,
)
from keras.layers import GlobalAveragePooling2D
import matplotlib.pyplot as plt
import seaborn as sns

# Setting Hyperparameters

Before training the neural network, several hyperparameters are defined.

- **Batch Size:** Number of training samples processed before updating model weights.
- **Number of Classes:** CIFAR-10 contains 10 object categories.
- **Epochs:** Number of complete passes through the training dataset.

These parameters directly influence training speed and model performance.

In [4]:
batch_size = 32  # The default batch size of keras.
num_classes = 10  # Number of class for the dataset
epochs = 100

# Loading the CIFAR-10 Dataset

The CIFAR-10 dataset is automatically downloaded (if not already present) and split into:

- Training set
- Testing set

Each image has:

- Resolution: **32 × 32 pixels**
- Channels: **3 (RGB)**
- Total Classes: **10**

The dataset contains:

- 50,000 training images
- 10,000 testing images

In [5]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
x_train shape: (50000, 32, 32, 3)
y_train shape: (50000, 1)
50000 train samples
10000 test samples


# Data Preprocessing

Neural networks perform better when input values are normalized.

The preprocessing steps include:

1. Converting image pixel values from integers to floating-point numbers.
2. Scaling pixel values from **0–255** to **0–1** by dividing by 255.
3. One-hot encoding the class labels using `to_categorical()`.

One-hot encoding converts labels into binary vectors suitable for multi-class classification.

In [6]:
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [7]:
x_train.shape

(50000, 32, 32, 3)

# Building the Convolutional Neural Network

A Sequential CNN model is constructed using multiple feature extraction and classification layers.

The architecture consists of:

- Input Layer
- Convolution Layers
- ReLU Activation Functions
- Max Pooling Layers
- Dropout Layers
- Global Average Pooling Layer
- Fully Connected Dense Layer
- Softmax Output Layer

The model progressively learns simple features (edges, textures) followed by more complex patterns (objects and shapes).

In [12]:
model = Sequential()
model.add(Input((32, 32, 3)))

# model.add(RandomFlip("horizontal"))
# model.add(RandomZoom(0.1))
# CONV => RELU => CONV => RELU => POOL => DROPOUT
model.add(Conv2D(32, (3, 3), padding='same',input_shape=x_train.shape[1:]))
model.add(Activation('relu'))
model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# CONV => RELU => CONV => RELU => POOL => DROPOUT
model.add(Conv2D(64, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# FLATTERN => DENSE => RELU => DROPOUT
model.add(GlobalAveragePooling2D())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.5))
# a softmax classifier
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 30, 30, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 15, 15, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 15, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 13, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │        33,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,978 (406.16 KB)

 Trainable params: 103,978 (406.16 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
opt = keras.optimizers.RMSprop(learning_rate=0.0001)
model.compile(loss='categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

# Training the CNN

The model is trained using the training dataset.

During training:

- Images are processed in batches.
- Model weights are updated after each batch.
- Validation accuracy and validation loss are computed using the test dataset after every epoch.
- The training data is shuffled before each epoch to improve generalization.

The `history` object stores all training and validation metrics for later visualization.

In [14]:
history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),
    shuffle=True
)

Epoch 1/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.2013 - loss: 2.0755 - val_accuracy: 0.2694 - val_loss: 1.9236
Epoch 2/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.2733 - loss: 1.8672 - val_accuracy: 0.3180 - val_loss: 1.7765
Epoch 3/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.3094 - loss: 1.7863 - val_accuracy: 0.3296 - val_loss: 1.7593
Epoch 4/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.3484 - loss: 1.7098 - val_accuracy: 0.3707 - val_loss: 1.6579
Epoch 5/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.3696 - loss: 1.6549 - val_accuracy: 0.3914 - val_loss: 1.6229
Epoch 6/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.3892 - loss: 1.6158 - val_accuracy: 0.3879 - val_loss: 1.6338
Epoch 7/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.4057 - loss: 1.5767 - val_accuracy: 0.4326 - val_loss: 1.5221
Epoch 8/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.4252 - loss: 

In [11]:
scores = model.evaluate(x_test, y_test, verbose=1)
print('Test loss:', scores[0])
print('Test accuracy:', scores[1])

pred = model.predict(x_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7392 - loss: 0.7605
Test loss: 0.7604900598526001
Test accuracy: 0.7391999959945679
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
